#### SG2 swerve base keyboard teleop — `real` (JSON)

VR·Joint·EEF와 같은 strict JSON worker를 사용합니다. `ri_motion_v5_py312` 커널에서 실행하세요.

```text
PC ── sg2_base_command JSON :5561 ──▶ inbound.py ──▶ /cmd_vel
PC ◀─ strict state + bridge ACK :5560 ── outbound.py
```

**로봇 준비**
- `VR_teleoperation/package/worker/`의 최신 `inbound.py`, `outbound.py`, `fixed_quest_protocol.py`를 배포하고 두 worker를 재시작합니다.
- JSON이 기본값입니다. `SG2_FIXED_QUEST=0`은 사용하지 않습니다.
- inbound의 `SG2_ZMQ_SUB_IP`에는 아래 셀이 출력하는 PC IP를 지정합니다.
- Base JSON 기능을 지원하지 않는 이전 worker에서는 ON이 거부됩니다.

**조작**
- Robot command는 OFF로 시작합니다. ON → 0 속도 arming·ACK → 이동 키 전체 해제 후 새 입력으로 움직입니다.
- W/S 전후, A/D 좌우, Q/E 회전. SPACE는 즉시 OFF로 전환합니다. 다시 움직이려면 ON을 선택합니다.
- OFF·창 닫기·오류에서는 0 stop을 보내고 명령 포트를 해제합니다. 통신·피드백 만료 시 worker의 최대 0.2초 lease watchdog이 0을 발행합니다.
- 평면 속도 크기는 최대 0.3 m/s, 회전은 최대 0.6 rad/s입니다. Joint/EEF/VR과 제어 세션을 동시에 소유할 수 없습니다.
- 화면 위치는 명령을 적분한 open-loop 표시이며 실제 위치 피드백이 아닙니다. 처음에는 Speed=SLOW로 주변 여유를 확인하세요.


In [6]:
import os
if "__vsc_ipynb_file__" in globals(): os.chdir(os.path.dirname(__vsc_ipynb_file__))   # VS Code 커널 cwd 보정
# Locate the shared workspace from a notebook, project or repository directory.
from pathlib import Path
import runpy

_notebook_dir = (Path(__vsc_ipynb_file__).resolve().parent
                 if "__vsc_ipynb_file__" in globals() else Path.cwd())
_search_bases = [_notebook_dir, *_notebook_dir.parents, Path.cwd(), *Path.cwd().parents]
_candidates = [candidate for base in _search_bases
               for candidate in (base, base / "VR_teleoperation")]
_root = next((candidate for candidate in _candidates
              if (candidate / ".project_v2").is_file()), None)
if _root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_bootstrap = runpy.run_path(str(_root / "package" / "init_project.py"))
for _name in _bootstrap["__all__"]:
    globals()[_name] = _bootstrap[_name]

import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
from utils import *
from package.base_control import *
from package.robot_client import SG2BaseCommandClient

app = get_qt_app()


[VR_teleoperation / ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation


In [7]:
PUB_TO_SG2_PORT = 5561
PUB_TO_SG2_IP = get_ip_address()
ROBOT_IP = "192.168.6.2"
SUB_FROM_SG2_PORT = 5560
SEND_HZ = 30
print("PC IP:", PUB_TO_SG2_IP, "→ inbound SG2_ZMQ_SUB_IP=%s" % PUB_TO_SG2_IP)
print("Base JSON feedback: tcp://%s:%d" % (ROBOT_IP, SUB_FROM_SG2_PORT))


PC IP: 192.168.6.103 → inbound SG2_ZMQ_SUB_IP=192.168.6.103
Base JSON feedback: tcp://192.168.6.2:5560


In [8]:
xml_path = merge_mjcfs(
    included_mjcf_files = [
        str(FLOOR_XML_PATH),
        str(SG2_XML_PATH),
    ],
    output_xml_path = str(SG2_SCENE_XML_PATH),)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=False)
env.reset()
base_z = get_sg2_base_z(env)
print("steer joints:",SG2_BASE_STEER_JOINT_NAMES)
print("drive joints:",SG2_BASE_DRIVE_JOINT_NAMES)

[merge_mjcfs] Merging [2] MJCF files:
  - [0] [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/xml/floor_simple_white.xml]
  - [1] [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/xml/ffw_sg2.xml]
[merge_mjcfs] Saved merged XML to:[/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/xml/scene_ffw_sg2_metaquest3.xml]
MuJoCo model loaded from [/home/vai/Desktop/yujin/VAI_AIWORKER/VR_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].
steer joints: ['left_wheel_steer_joint', 'right_wheel_steer_joint', 'rear_wheel_steer_joint']
drive joints: ['left_wheel_drive_joint', 'right_wheel_drive_joint', 'rear_wheel_drive_joint']


In [9]:
env.init_viewer(title="SG2 Base Teleop (real)",x_offset=0.23,y_offset=0.00,width=0.77,height=1.0,fontscale=180)
env.viewer.set_cam_info(200.0,4.0,-35.0,[0.0,0.0,0.8])

teleop = SG2BaseTeleop()
ctrl_radio = create_sg2_base_command_widget(real=True)
input_gate = SG2BaseInputGate()

In [ ]:
hz = 30
tmr = SimpleTimer(name="Base Real", Hz=hz, verbose=False)
tmr_send = SimpleTimer(name="Base JSON", Hz=SEND_HZ, verbose=False)
last_robot_status = {"state": "OFF", "reason": "control disabled", "acknowledged_seq": -1}
prev_on_flag = False
pub_robot = None
try:
    pub_robot = SG2BaseCommandClient(robot_ip=ROBOT_IP, state_port=SUB_FROM_SG2_PORT,
                                     command_port=PUB_TO_SG2_PORT)
    tmr.start(); tmr_send.start(); env.reset_wall_time(); env.reset_sim_time()
    while env.is_viewer_alive() and ctrl_radio.isVisible():
        app.processEvents()
        space_stop = env.is_key_pressed_once(key=glfw.KEY_SPACE)
        if space_stop:
            ctrl_radio.set_button_option("Robot command", "OFF")
        on_flag = ctrl_radio.get_button_option(label_text="Robot command") == "ON"
        if space_stop or on_flag != prev_on_flag:
            teleop.stop()
            input_gate.waiting_for_release = True
            if not on_flag:
                last_robot_status = pub_robot.step([0., 0., 0.], enabled=False)
            prev_on_flag = on_flag

        if tmr.do_run():
            teleop.scale = SG2_BASE_SPEED_SCALES[ctrl_radio.get_button_option(label_text="Speed")]
            input_gate.read_keys(teleop, env, enabled=on_flag,
                                 active=last_robot_status["state"] == "ACTIVE")
            teleop.update(1.0 / hz)
            apply_sg2_base_to_env(env, teleop, base_z)
            if ctrl_radio.get_button_option(label_text="Camera") == "FOLLOW":
                env.viewer.set_cam_info(lookat=[teleop.pose[0], teleop.pose[1], 0.8])
            env.plot_global_coordinate_axes()
            draw_sg2_base(env, teleop, base_z)
            status = "%s  ACK %s  %s" % (last_robot_status["state"],
                      last_robot_status["acknowledged_seq"], last_robot_status["reason"])
            if input_gate.waiting_for_release and on_flag:
                status += " | release motion keys before driving"
            env.viewer_text_overlay("Base", make_sg2_base_overlay_text(teleop, extra=status), loc="top left")
            env.render()
            tmr.end()

        if tmr_send.do_run():
            last_robot_status = pub_robot.step(teleop.v_cmd, enabled=on_flag)
            if last_robot_status["state"] != "ACTIVE":
                teleop.stop()
            if last_robot_status["state"] == "FAULT":
                ctrl_radio.set_button_option("Robot command", "OFF")
                print("[Base JSON] FAULT:", last_robot_status["reason"])
            tmr_send.end()
except KeyboardInterrupt:
    print("Interrupted.")
finally:
    teleop.stop()
    try:
        if pub_robot is not None:
            pub_robot.close()
    finally:
        try:
            ctrl_radio.close()
        finally:
            env.close_viewer()
    print("Done. Base stop requested; worker lease watchdog remains active.")


[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_required; cleanup: strict_command_envelope_required
[Base JSON] FAULT: strict_command_envelope_require